# Medication Review AI: interactive runner

This notebook is the interactive entry point for inspecting the real production functions. Jupyter runs Python cells in a persistent kernel, like a REPL: variables and results remain available until the kernel is restarted. The application imports the same production modules; this notebook provides manual controls and inspection.

**Run order:** select the project's `.venv` kernel, run section 0, choose inputs in section 1A, then choose switches and parameters in section 1B. Run section 2 to prepare/check resources, section 3 for rules, then the enabled sections you want to inspect. Use Run All only after reviewing the controls. The default switches enable retrieval, rule-based patient review and retrieval-only evaluation. Enable model calls explicitly when API access is available. Section 2 can download missing data and rebuild a mismatched index.

| Goal | Inputs and sections |
| --- | --- |
| Prepared patient scenario | `input_mode="scenario"`, choose `scenario_id`; run sections 2, 3 and 6. |
| Custom patient review | `input_mode="custom"`, choose patient, leave `custom_doctor_question=""`; run sections 2, 3 and 6. |
| Patient plus question / proposed medicine | Custom mode with `custom_doctor_question`; section 6 requires `PIPELINE_USE_LLM=True`. |
| Standalone medicine question | Set `rag_task="general"` and `search_question`; use `nregistro=None`, `sections=None` for an unfiltered search; run sections 2, 4 and 5. These requests do not use the selected patient. |
| Batch RAG evaluation | Set `evaluation_split` and evaluation switches; run section 7 after preparing the required sources/index. |

Disable unrelated `RUN_*` switches when using Run All. Section 3 always runs rules, and section 2 always performs resource preparation when its cell is executed. The notebook still loads a patient for its patient-inspection stages even when you only want to run the standalone RAG stages.

From the repository root:
```powershell
.venv\Scripts\python -m jupyterlab MAIN.ipynb
```
Install the dependencies with `uv pip install -r requirements.txt` if needed. Groq generation needs the API key in `.env`; retrieval needs the local index and E5 model. Use synthetic patients. Rerun section 1 and dependent stages after changing inputs to avoid inspecting stale results. Restart the kernel after editing imported Python modules.


## 0. Imports and display

Locate the repository, make its `src` package importable and load the functions used by later cells. `Path.cwd()` starts from the notebook's working directory; `ROOT` should resolve to the project root.

**Functions and files**
- This cell imports Python and IPython display utilities. Project functions are imported here; section 1 supplies inputs to their calls.
- Project paths come from `src/config.py`; selected experiment parameters and execution switches are set in section 1.
- `show_fragments()` is a display helper defined in this cell. It prints each fragment's complete text, rank, medicine, section and distance. `JSON`, `Markdown` and `display()` come from `IPython.display` and control presentation.

**Choices:** select the project's `.venv` kernel and open `MAIN.ipynb` from the repository root. Later cells depend on these imports. Rerun section 1 and subsequent cells after changing experiment settings. Restart the kernel after editing project source code.

In [ ]:
import json
import sys
from pathlib import Path

from IPython.display import JSON, Markdown, display

ROOT = next(
    (folder for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (folder / "src" / "main.py").is_file()
     and (folder / "data" / "pacientes" / "patient_scenarios.json").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open MAIN.ipynb from the repository root or one of its subfolders.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import PATIENTS_DIR, PRESCRIPTIONS_FILE
from src.main import load_catalogue, load_rule_data, review_patient, review_request
from src.Reports.informe import Patient, generate_report
from src.Evaluation.evaluar import evaluate_cases

def show_fragments(fragments):
    """Show complete source text, with its rank and metadata."""
    if not fragments:
        print("No fragments returned. This does not establish safety.")
    for rank, fragment in enumerate(fragments, 1):
        display(JSON({"rank": rank, **{
            key: fragment.get(key)
            for key in ("id", "nregistro", "nombre", "seccion", "titulo", "distance")
        }}, expanded=True))
        print(fragment["texto"])
        print()

print("Repository:", ROOT)

## 1. Select the test and experiment settings

### 1A. Patient scenario or custom inputs

Choose `input_mode="scenario"` to load one of the 12 entries in `data/pacientes/patient_scenarios.json` using `scenario_id`. This loads its patient file, question and any clinical-context overrides. Overrides apply in memory. The 12 scenarios use 9 distinct patients; some patients appear in several situations.

Choose `input_mode="custom"` to set `custom_patient_file`, `custom_doctor_question` and optional `custom_patient_overrides` yourself. A blank question requests a standard review. The cell displays the selected scenario, patient, medicines, clinical context, question and expected scenario checks so the current test is easy to identify.

**Functions:** `json.loads()` reads scenario/patient data; `Patient.model_validate()` validates patient inputs; `load_catalogue()` loads medicine metadata.

**Standalone RAG inputs:** `rag_task`, `search_question`, `nregistro`, `sections` and `ingredient` configure sections 4-5 independently. Patient scenarios configure sections 3 and 6; they do not automatically become standalone RAG requests. The summary prints only the selected patient-workflow inputs; standalone RAG inputs are inspected in section 4.

Run the input cell, then the controls below and subsequent sections whenever you change the selected test.

For an unrestricted standalone question, set `rag_task="general"`, type `search_question`, and set `nregistro=None`, `sections=None`. Interaction mode requires a specific medicine registration and another ingredient; it fixes retrieval to 4.5. Prepared patient scenarios do not change these standalone controls.


In [ ]:
# Prepared Patient-workflow input: "scenario" or "custom".
input_mode = "custom"


# =================================================
# Prepared Patient-Questions Cases
scenario_id = "P-E2"  # Used only in scenario mode; available IDs are printed below.
# =================================================


# =================================================
# Custom patient and question (used only in custom mode).
custom_patient_file = PATIENTS_DIR / "paciente_A.json"
custom_doctor_question = ""  # Blank requests a standard medication review.
custom_patient_overrides = {}  # Example: {"contexto_clinico": "Synthetic context"}

# Standalone RAG experiment for sections 4-5, independent of the patient scenario.
rag_task = "general"  # "general" or "interaction"
search_question = "¿Qué interacciones se describen con warfarina?"
nregistro = "59718"  # None searches all medicines in general mode.
sections = ["4.5"]  # None searches all sections in general mode.
ingredient = "warfarina"  # Used in interaction mode.
# =================================================


scenarios_file = PATIENTS_DIR / "patient_scenarios.json"
scenarios = json.loads(scenarios_file.read_text(encoding="utf-8"))
scenario_by_id = {item["id"]: item for item in scenarios}
selected_scenario = None

if input_mode == "scenario":
    if scenario_id not in scenario_by_id:
        raise ValueError(f"Unknown scenario {scenario_id!r}. Choose from: {', '.join(scenario_by_id)}")
    selected_scenario = scenario_by_id[scenario_id]
    patient_file = ROOT / selected_scenario["patient_file"]
    doctor_question = selected_scenario["doctor_question"]
    patient_overrides = selected_scenario.get("patient_overrides", {})
elif input_mode == "custom":
    patient_file = Path(custom_patient_file)
    doctor_question = custom_doctor_question
    patient_overrides = custom_patient_overrides
else:
    raise ValueError('input_mode must be "scenario" or "custom".')

patient_data = json.loads(patient_file.read_text(encoding="utf-8"))
patient_data.update(patient_overrides)  # In-memory changes only.
patient = Patient.model_validate(patient_data)
catalogue = load_catalogue()

print("PATIENT WORKFLOW (sections 3 and 6)")
print("Mode:", input_mode)
if selected_scenario:
    print("Scenario:", selected_scenario["id"])
print("Patient:", patient.id, "| Age:", patient.edad, "| File:", patient_file.name)
print("Question:", doctor_question or "[Blank: standard medication review]")
if patient.contexto_clinico:
    print("Clinical context:", patient.contexto_clinico)
print("Medicines:")
for medicine in patient.medicamentos:
    print(f"  - {medicine.nombre} ({medicine.nregistro}) | {medicine.dosis} | {medicine.prescriptor}")
if selected_scenario:
    print("Expected checks:")
    for check in selected_scenario.get("manual_checks", []):
        print("  -", check)
if input_mode == "scenario":
    print("\nAvailable scenario IDs:", ", ".join(scenario_by_id))


### 1B. Run controls and experiment parameters

Set `k` and choose which stages to execute. `RUN_RETRIEVAL` prepares section 4; `RUN_LLM` generates section 5; `RUN_PIPELINE` runs section 6, with `PIPELINE_USE_LLM` enabling its model calls. A nonblank patient question requires model calls.

`RUN_EVALUATION` and `EVALUATION_USE_LLM` control section 7, which uses `evaluation_questions.json` and `evaluation_split` independently of the selected patient scenario. `case_index` chooses the batch result to inspect. Data preparation controls are `check_updates_since` and `rebuild_index`. These settings do not modify project configuration defaults. Rerun this cell and relevant later sections after changing parameters.

`generation_options={}` uses project model defaults. To compare generation settings, use for example `{"model": "openai/gpt-oss-120b", "temperature": 0.0}`. These options are passed to standalone answers, patient-question planning/checks/answers, and batch generation. They have no effect on rules or retrieval-only experiments. `k` must be at least 1. `case_index` is a zero-based index into the selected batch results.


In [ ]:
# Retrieval limit shared by standalone RAG, the patient workflow and evaluation.
k = 5
generation_options = {}  # Optional model / temperature overrides; {} uses defaults.

# Select which parts to run.
RUN_RETRIEVAL = True
RUN_LLM = False
RUN_PIPELINE = True
PIPELINE_USE_LLM = False
RUN_EVALUATION = True
EVALUATION_USE_LLM = False

# Data preparation and batch evaluation (independent of the selected scenario).
check_updates_since = None  # Optional snapshot date: "dd/mm/yyyy".
rebuild_index = False
cases_file = ROOT / "src" / "Evaluation" / "evaluation_questions.json"
evaluation_split = "development"
case_index = 0

print("k:", k, "| Batch evaluation split:", evaluation_split)
if doctor_question.strip() and RUN_PIPELINE and not PIPELINE_USE_LLM:
    print("Note: the selected patient question requires PIPELINE_USE_LLM=True.")


## 2. Prepare data and check coverage

Prepare the resources needed for the selected medicines, then display which resources were reused, downloaded or built. The scope combines the patient's medicines, `nregistro` when specified, and catalogue entries marked as having a technical sheet.

**Functions and files:** `prepare_notebook_data(registrations, check_updates_since=...)` in `src/Data/ingesta.py` coordinates:
1. `load_cascades()` from `src/Cascades/reglas.py`: validate the curated `data/cascadas.json`.
2. `ensure_nomenclator()`: reuse usable `prescriptions.json`, or prepare it from local XML/ZIP. It calls `download_zip()` / `unpack_zip()` from `src/Data/download_nomenclator.py` when required, and `main()` from `src/Data/parse_nomenclator.py` to parse the XML.
3. `ensure_fragments()`: use `load_fragments()` to check JSON shape, fragment fields and duplicate section IDs. For registrations absent from that file, call `chunks_for_medicine()` from `src/Data/parse_cima.py`. Successful downloads are added to the existing file; incomplete downloads are reported.
4. `ensure_index()`: compare Chroma's stored IDs, original text and metadata with the local fragments. A mismatch or missing collection triggers `main()` from `src/Rag/indice.py` to generate embeddings and build the collection. Cached search resources are then cleared.
5. If requested, `check_cima_updates()` calls `fetch_json()` from `src/Data/parse_cima.py` to read all pages of CIMA's `registroCambios` endpoint.

**Choices**
- `registrations` determines which medicines must have source coverage. Edit this set if you deliberately want a different preparation scope.
- `check_updates_since = None` skips remote update checks. A date string in `dd/mm/yyyy` format reports changes since that source snapshot date. Existing covered medicines are not refreshed automatically.
- Paths and embedding settings come from `src/config.py`; the optional `rebuild_index` switch is in section 1. Set that switch after changing the embedding recipe, preferably using a separate database folder. Otherwise the index check compares stored contents rather than the model that created the vectors.

**Expected output:** preparation messages, still-missing registrations, fragment counts, medicine coverage and any requested change records. Coverage means at least one stored fragment for a medicine; it does not establish that every possible section is present. Invalid local files raise an error for inspection. The curated cascade file must be restored from Git if missing.

In [4]:
from src.Data.ingesta import load_fragments, prepare_notebook_data

registrations = {medicine.nregistro for medicine in patient.medicamentos}
registrations.update(entry.nregistro for entry in catalogue.values() if entry.ficha)
if nregistro:
    registrations.add(nregistro)

preparation = prepare_notebook_data(
    sorted(registrations), check_updates_since=check_updates_since, rebuild_index=rebuild_index,
)
display(JSON(preparation, expanded=True))

source_fragments = load_fragments()
covered = {fragment["nregistro"] for fragment in source_fragments}
display(JSON({
    "fragments": len(source_fragments),
    "registrations": sorted(covered),
    "patient_medicines_without_parsed_sources": [
        medicine.nregistro for medicine in patient.medicamentos
        if medicine.nregistro not in covered
    ],
    "patient_medicines_outside_catalogue": sorted(
        {medicine.nregistro for medicine in patient.medicamentos} - catalogue.keys()
    ),
}, expanded=True))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## 3. Rules: duplicates, ATC families and cascade hypotheses

Convert the patient's registered medicines into ingredient and ATC codes, then identify shared ingredients, shared therapeutic families and matching prescribing-cascade patterns.

**Functions and files**
- `load_rule_data()` in `src/main.py` calls `load_drug_info()` and `load_cascades()` in `src/Cascades/reglas.py`. Package-level nomenclator records are combined into medicine-level code lists.
- `review_patient(patient, catalogue, rule_data=rule_data, use_model=False)` in `src/main.py` coordinates the checks and converts their results into report findings.
- Its rules branch calls `build_medications()`, `find_duplicates()`, `find_family_warnings()` and `find_cascades()` from `src/Cascades/reglas.py`. Family warnings omit pairs already classified as ingredient duplicates.

**Choices:** change `custom_patient_overrides["medicamentos"]` in custom mode (supply the complete replacement medicine list), or select another patient/scenario in section 1 to explore different combinations. Cascade definitions live in `data/cascadas.json`; matching logic and the ATC-family prefix length live in `src/Cascades/reglas.py`. The `use_model=False` argument selects a rules-only review. If `rule_data` is unavailable, the coordinator falls back to catalogue ingredient-name comparisons for duplicates and marks ATC/cascade checks as unavailable.

**Expected output:** each finding's type, description, references and suggested clinician question, followed by unavailable checks. Cascade matches are co-occurrence hypotheses to investigate, not evidence that one prescription caused another. Doses and clinical context are patient information; these rules do not calculate dose-dependent risk.

In [5]:
rule_data = load_rule_data() if PRESCRIPTIONS_FILE.exists() else None
rule_result = review_patient(patient, catalogue, rule_data=rule_data, use_model=False)
display(JSON({"findings": [finding.model_dump(mode="json") for finding in rule_result.findings]}, expanded=True))
display(JSON({
    "not_checked": sorted(rule_result.unchecked),
    "could_not_check": rule_result.could_not_check,
}, expanded=True))

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## 4. Retrieve fragments and inspect the exact prompt

Search the vector index, select evidence and build the messages that will be sent for generation. This lets you inspect the source material and instructions before assessing an answer.

**Functions and files:** both preparation functions come from `src/Rag/rag.py`.
- General mode calls `prepare_general_search(search_question, nregistro, sections, k=k)`.
- Interaction mode calls `prepare_interaction_search(nregistro, ingredient, k=k)`. It constructs a Spanish interaction question, searches section `4.5` and retains fragments containing the normalised ingredient text.
- Both use `search_fragments()` in `src/Rag/busqueda.py`. `load_search_resources()` opens the Chroma collection and loads E5; the query is embedded with the `query: ` prefix and ranked within the eligible sources.
- `build_messages()` in `rag.py` combines the selected task prompt, question and retained source text.

**Choices:** set `RUN_RETRIEVAL=True`, then vary the section 1 inputs. General mode accepts medicine and section filters; section `4.5` includes `4.5.1` but not `4.50`. `sections=None` includes all sections; an empty list is invalid. Interaction mode uses `ingredient` rather than `search_question` and fixes the section filter. Larger `k` gives the retrieval stage more candidates but may introduce irrelevant text.

**Expected output:** preparation status (`ready` or `not_found`), actual question, complete retained fragments and system/user messages. Lower cosine distance indicates a closer vector match; it is not an accuracy or confidence percentage. An ingredient mention is a text filter, not proof of an interaction. This standalone search uses the query and filters, without the patient sheet.

In [6]:
prepared = None
if RUN_RETRIEVAL:
    from src.Rag.rag import prepare_general_search, prepare_interaction_search

    if rag_task == "interaction":
        prepared = prepare_interaction_search(nregistro, ingredient, k=k)
    elif rag_task == "general":
        prepared = prepare_general_search(search_question, nregistro, sections, k=k)
    else:
        raise ValueError("rag_task must be general or interaction.")
    print("Preparation status:", prepared["status"])
    print("Actual search question:", prepared["question"])
    show_fragments(prepared["fragments"])
    for message in prepared["messages"]:
        print("\n---", message["role"].upper(), "---")
        print(message["content"])
else:
    print("Retrieval disabled. Enable RUN_RETRIEVAL to inspect the index.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Preparation status: ready
Actual search question: ¿Qué interacciones se describen con warfarina?


<IPython.core.display.JSON object>

Medicamentos contraindicados
Inhibidores de la monoaminooxidasa (IMAOs)
IMAOs irreversibles (ej. selegilina)
Sertralina no se puede utilizar en combinación con IMAOs irreversibles tales como la selegilina. No debe iniciarse el tratamiento con sertralina hasta al menos 14 días después de haber suspendido el tratamiento con un IMAO irreversible. El tratamiento con sertralina se debe suspender al menos 7 días antes de iniciar el tratamiento con un IMAO irreversible (ver sección 4.3).
Inhibidor reversible selectivo de la MAO-A (moclobemida)
Debido al riesgo de síndrome serotoninérgico, no debe administrarse sertralina en combinación con un IMAO reversible y selectivo, como la moclobemida. Tras el tratamiento con un inhibidor reversible de la MAO, puede establecerse un periodo de retirada más corto, inferior a 14 días, antes de comenzar el tratamiento con sertralina. Se recomienda que el tratamiento con sertralina se suspenda al menos 7 días antes de iniciar el tratamiento con un IMAO rever

### Inspect fragments removed by the interaction filter

For an interaction request, call `search_fragments()` from `src/Rag/busqueda.py` with the prepared question, selected `nregistro`, `sections=["4.5"]` and `k`. Compare the returned IDs with those retained by `prepare_interaction_search()`.

**Choices:** change `ingredient`, `nregistro` or `k` in section 1 and rerun section 4. The ingredient is matched after case, accent and whitespace normalisation; The filter also accepts verified ATC class terms where available.

**Expected output:** the full text of fragments removed by the mention filter. An empty list means no retrieved fragments were removed. This separates a retrieval miss from evidence excluded after retrieval.

In [7]:
if prepared is not None and rag_task == "interaction":
    from src.Rag.busqueda import search_fragments

    before_filter = search_fragments(
        prepared["question"], nregistro=nregistro, sections=["4.5"], k=k,
    )
    retained_ids = {fragment["id"] for fragment in prepared["fragments"]}
    print("Removed by the ingredient filter:")
    show_fragments([fragment for fragment in before_filter if fragment["id"] not in retained_ids])

## 5. Generate and verify a standalone RAG answer

Generate a structured answer from the retrieved sources, then inspect whether it passes the response and evidence checks.

**Functions and files**
- General mode calls `run_prepared_request(prepared)` in `src/Rag/rag.py`, using the messages inspected in section 4.
- Interaction mode calls `review_interaction(nregistro, ingredient, k=k)` in `rag.py`. It prepares the search again, generates the answer and requires an explicit interaction classification for a `found` result.
- Generation calls `llm_medication_review()` in `src/llm.py`, which uses the Groq client and validates JSON against `StructuredOutputRAG`. Invalid JSON/schema output gets one fresh generation attempt.
- The RAG checks compare generated text with `FORBIDDEN_PHRASES` from `src/config.py` and call `verificar_cita()` from `src/Rag/verificacion.py` for every citation's medicine, exact section and quotation text.

**Choices:** set `RUN_LLM=True` after preparing section 4. Change the query, medicine, sections, ingredient or `k` upstream to change the evidence. Task instructions are `GENERAL_SYSTEM_PROMPT` and `INTERACTION_SYSTEM_PROMPT` in `rag.py`. Generation uses `generation_options` from section 1B over the defaults in `src/config.py`; citation and wording checks use shared project settings. Credentials remain in `.env`. Rerun from section 1 after editing experiment inputs.

**Expected output:** `status`, `reason` and the accepted `response`, including description and citations.
- `found`: an answer with evidence; interaction mode additionally distinguishes `interaction_detected=True` from a documented `False`.
- `not_found`: missing or insufficient supporting evidence, never a safety conclusion.
- `rejected`: invalid structure, forbidden wording, an unverifiable citation or a missing required classification.
- `error`: generation could not complete.

Rejected answer text is withheld. Literal citation checks establish that quoted text exists in a supplied source; relevance and whether the quotation supports the conclusion still require inspection.

In [8]:
rag_result = None
if RUN_LLM:
    if prepared is None:
        raise RuntimeError("Enable RUN_RETRIEVAL and run section 4 first.")
    from src.Rag.rag import run_prepared_request, review_interaction

    if rag_task == "interaction":
        rag_result = review_interaction(nregistro, ingredient, k=k, generation_options=generation_options)
    else:
        rag_result = run_prepared_request(prepared, generation_options=generation_options)
    display(JSON({
        key: rag_result.get(key) for key in ("status", "reason", "response")
    }, expanded=True))
else:
    print("Standalone generation disabled.")

<IPython.core.display.JSON object>

## 6. Complete patient workflow and report

Process the patient sheet and optional doctor's question together, run the relevant checks, and assemble the final readable report.

**Functions and files:** this cell calls `review_request(patient, catalogue, doctor_question, rule_data=rule_data, use_model=PIPELINE_USE_LLM)` in `src/main.py`.
1. A blank question routes to `review_patient()`. With model checks enabled, the coordinator plans eligible medicine-pair searches and calls `review_interaction()` from `src/Rag/rag.py`.
2. A nonblank question first calls `llm_prepare_question()` from `src/llm.py`. It resolves catalogue medicines and plans an overall review, a medicine question, a proposed new medicine, or a clarification.
3. The coordinator calls `review_patient()` or `check_new_medicine()` from `main.py`. A new-medicine check focuses on pairs involving the proposed product.
4. `answer_patient_question()` in `src/Rag/rag.py` retrieves scoped evidence and calls `llm_answer_patient_question()` in `src/llm.py` to answer using the patient features, checked findings and limitations. Citations and referenced findings are checked before release.
5. `generate_report()` in `src/Reports/informe.py` builds report blocks. The `Report.markdown` property uses `render_markdown()` from that file for display.

**Choices**
- `RUN_PIPELINE=True` executes this section.
- In custom mode, `custom_doctor_question=""` requests a standard review. A question about an existing or proposed medicine requests a focused answer; unresolved medicine identities can produce a clarification.
- `PIPELINE_USE_LLM=False` selects available rule checks for a blank question. A nonblank question requires `True` to be interpreted and answered.
- Change the patient's medicines and optional `contexto_clinico` in section 1 to change the supplied patient facts.

The standalone `search_question`, `rag_task`, `sections` and `ingredient` describe a separate request. This workflow plans searches from `doctor_question` and the patient sheet. The call passes section 1B's `k` and `generation_options` through the coordinator to the applicable searches and model calls. Shared-ingredient pairs are searched as well as receiving duplicate warnings.

**Expected output:** clarification when needed; otherwise an audit log, searches without evidence, incomplete checks and the rendered report with findings and citations. `safety_flags` identifies report text withheld by the report-level wording check.

In [9]:
pipeline_result = None
if RUN_PIPELINE:
    pipeline_result = review_request(
        patient, catalogue, doctor_question,
        rule_data=rule_data, use_model=PIPELINE_USE_LLM,
        k=k, generation_options=generation_options,
    )
    print("Clarification:", pipeline_result.clarification)
    checks = pipeline_result.checks
    display(JSON({
        "audit_log": checks.log,
        "no_evidence": checks.checked_without_evidence,
        "could_not_check": checks.could_not_check,
        "not_checked": sorted(checks.unchecked),
    }, expanded=True))
    if not pipeline_result.clarification:
        report = generate_report(
            patient, checks.findings,
            checked_without_evidence=checks.checked_without_evidence,
            could_not_check=checks.could_not_check,
            unchecked_types=checks.unchecked, proposed=pipeline_result.proposed,
            question=pipeline_result.question, answer=pipeline_result.answer,
            answer_citations=pipeline_result.citations,
            supporting_findings=pipeline_result.supporting_findings,
        )
        display(Markdown(report.markdown))
        print("Report safety flags:", report.safety_flags)
else:
    print("Complete workflow disabled.")

Clarification: None


<IPython.core.display.JSON object>

# Medication review report

**Patient:** A · 78 years · development patient  
**Generated:** 2026-11-27 11:37  
**Medicines reviewed:** 2 from 2 prescriber(s)

## Summary

| Check | Result |
|---|---|
| Duplicate active ingredients | 1 |
| Same therapeutic group (ATC) | 0 |
| Possible interactions | 2 |
| Possible prescribing cascades (hypotheses to review) | 0 |

## 1. Medicines by prescriber

### Dra. Martín (Traumatología) (1)

| Medicine | Dose | Reg. no. |
|---|---|---|
| Paracetamol Kern Pharma 1 g comprimidos EFG | 1 g cada 8 horas si hay dolor | 68339 |

### Dr. Ruiz (Atención Primaria) (1)

| Medicine | Dose | Reg. no. |
|---|---|---|
| Antidol 1 g comprimidos | 1 g cada 8 horas si hay dolor | 85780 |

## 2. Duplicate active ingredients

'Paracetamol Kern Pharma 1 g comprimidos EFG' (Dra. Martín (Traumatología)) and 'Antidol 1 g comprimidos' (Dr. Ruiz (Atención Primaria)) contain the same active ingredient. Possible duplicate to review.

## 3. Same therapeutic group (ATC)

None found.

## 4. Possible interactions

_Information retrieved from the official technical sheets (CIMA). Every quotation below was checked against the section it cites._

### Finding 4.1

Se advierte una posible interacción entre paracetamol y flucloxacilina que puede asociarse a acidosis metabólica con desequilibrio aniónico alto, especialmente en pacientes con factores de riesgo.

**Source:** Paracetamol Kern Pharma 1 g comprimidos EFG (reg. 68339), section 4.5 (Interaction with other medicinal products)

> “Se debe tener precaución cuando se utiliza paracetamol de forma concomitante con flucloxacilina, ya que la administración concurrente se ha asociado con acidosis metabólica con desequilibrio aniónico alto debido a acidosis piroglutámica, especialmente en pacientes con factores de riesgo (ver sección 4.4).”

### Finding 4.2

El paracetamol puede incrementar el efecto anticoagulante de los anticoagulantes orales (acenocumarol, warfarina) cuando se administra crónicamente a dosis superiores a 2 g/día, probablemente por una disminución de la síntesis hepática de los factores de coagulación.

**Source:** Antidol 1 g comprimidos (reg. 85780), section 4.5 (Interaction with other medicinal products)

> “Anticoagulantes orales (acenocumarol, warfarina): La administración crónica de dosis de paracetamol superiores a 2 g/día con este tipo de productos, puede provocar un incremento del efecto anticoagulante, posiblemente debido a una disminución de la síntesis hepática de los factores que favorecen la coagulación.”

## 5. Possible prescribing cascades (hypotheses to review)

_Matched against the group's curated cascade table. These are hypotheses for the clinician to review, not diagnoses._

None found.

## 6. Questions for the next appointment

1. Is the patient taking both Paracetamol Kern Pharma 1 g comprimidos EFG and Antidol 1 g comprimidos, and what total daily amount of the shared active ingredient does that add up to?
2. Have the prescribers involved (Dr. Ruiz (Atención Primaria) and Dra. Martín (Traumatología)) reviewed the combined use of Paracetamol Kern Pharma 1 g comprimidos EFG and Antidol 1 g comprimidos?
3. Is the patient taking any other medicines, supplements or herbal products that are not on this list?

---

*Automatically generated support information. Does not replace the judgement of a healthcare professional.*


Report safety flags: []


## 7. Repeatable RAG evaluation

Run a set of reviewed cases with the selected retrieval limit and compare expected evidence with actual results. This evaluates standalone RAG requests; section 6 is the separate patient-workflow inspection.

**Functions and files:** `json.loads()` reads the `cases_file` selected in section 1 (initially `src/Evaluation/evaluation_questions.json`). `evaluate_cases(cases, k=..., generate=..., generation_options=...)` in `src/Evaluation/evaluar.py`:
1. Validates case definitions and unique IDs.
2. Calls `prepare_general_search()` / `prepare_interaction_search()` from `src/Rag/rag.py` for retrieval-only experiments, or `answer_question()` / `review_interaction()` for generated answers.
3. Uses `score_retrieval()` and `summarise_results()` in `evaluar.py` to calculate labelled metrics and their denominators.
4. Retains each case's actual fragments, response, status, reason and elapsed time, together with run settings and a timestamp.

**Choices:** All controls are set in section 1. `RUN_EVALUATION=True` executes the selected cases; `EVALUATION_USE_LLM` controls answer generation. The evaluation uses the `k` defined in section 1. Change it and rerun to compare another limit. `evaluation_split="development"` selects tuning cases; `"evaluation"` selects reserved final cases. Cases without a `split` default to development. Batch questions and medicine filters come from each case, independently of the single-case inputs in section 1.

**Case fields**
- General case: `id`, `task: "general"`, `question`, and optional `nregistro` / `sections`.
- Interaction case: `id`, `task: "interaction"`, `nregistro` and `ingredient`.
- `expected_sources`: reviewed list of `{"nregistro": "...", "seccion": "..."}` source labels.
- `expected_status`: `"found"` or `"not_found"`, describing expected evidence availability.
- `expected_interaction_detected`: a reviewed `true` / `false` classification when the evidence supports one.
- `reference_answer`: reviewed text for manual comparison; the script does not score its wording.
- `split`: `"development"` or `"evaluation"`.

| Summary metric | Meaning |
| --- | --- |
| `source_hit` (Source Hit@k) | Fraction of source-labelled cases retrieving at least one expected medicine/section. |
| `source_recall` (Source Recall@k) | Average fraction of each case's expected medicine/section pairs retrieved. |
| `status_match` | Fraction of status-labelled generated cases with the expected evidence status. |
| `interaction_match` | Fraction of classification-labelled generated cases returning `found` with the expected interaction boolean. |

Missing labels remain unscored. Source labels match exact sections, so `4.5` and `4.5.1` are distinct. Retrieval exceptions are excluded from retrieval averages and counted as errors; read the denominators alongside the scores. Failed or rejected generation counts as a mismatch for supplied answer labels. These metrics do not establish answer correctness or clinical accuracy.

**Expected output:** a summary for the selected `k`, followed by inspectable case results. `evaluation_questions.json` contains 15 development and 5 reserved RAG cases. Inspect their source labels and reference answers alongside the generated results.
The cases are source-checked drafts for review. The reserved cases and existing evaluation patients have already been inspected and share concepts with development cases; they are not an untouched independent validation cohort. `benchmark_manifest.json` records source hashes but is not automatically checked here. Section 2 can change source coverage, so recheck affected labels after data updates.


In [10]:
cases = json.loads(cases_file.read_text(encoding="utf-8"))
cases = [case for case in cases if case.get("split", "development") == evaluation_split]
evaluation = None

if not cases:
    print("No cases match the selected evaluation_split; check the split and cases_file.")
elif RUN_EVALUATION:
    evaluation = evaluate_cases(cases, k=k, generate=EVALUATION_USE_LLM,
                                generation_options=generation_options)
    display(JSON(evaluation["summary"], expanded=True))
else:
    print("Evaluation disabled. Loaded cases:", len(cases))

<IPython.core.display.JSON object>

### Inspect an individual evaluation result

Select `evaluation["results"][case_index]` from the run above. `JSON()` displays its labels, status, answer, scores and timing; `show_fragments()` prints the full retrieved sources.

**Choices:** `case_index` in section 1 selects a case, starting at zero. Change `k` and rerun the evaluation to compare another configuration.

**Expected output:** an expected-versus-actual record and its evidence. Compare the reference answer and quotations manually to understand why a case succeeded or failed.

In [11]:
if evaluation is not None and evaluation["results"]:
    case_result = evaluation["results"][case_index]
    display(JSON({key: value for key, value in case_result.items() if key != "fragments"}, expanded=True))
    show_fragments(case_result["fragments"])

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

En general, los AINEs deben emplearse con precaución cuando se utilizan con otros fármacos que pueden aumentar el riesgo de ulceración gastrointestinal, hemorragia gastrointestinal o disfunción renal.
Debe evitarse el uso de ibuprofeno en combinación con:
Ácido acetilsalicílico (aspirina): en general, no se recomienda la administración concomitante de ibuprofeno y ácido acetilsalicílico debido a la posibilidad de que aumenten los efectos adversos.
Datos experimentales sugieren que ibuprofeno puede inhibir de forma competitiva el efecto de dosis bajas de ácido acetilsalicílico sobre la agregación plaquetaria cuando se administran de forma concomitante. Aunque hay ciertas dudas respecto a la extrapolación de estos datos a la situación clínica, la posibilidad de que el uso habitual a largo plazo de ibuprofeno pueda reducir el efecto cardioprotector de dosis bajas de ácido acetilsalicílico no puede excluirse. Se considera que es probable que no haya un efecto clínicamente relevante con el 